In [ ]:

# 1. Load Dataset

import pandas as pd

df = pd.read_csv(
    "../datasets/CMAPSSData/train_FD001.txt",
    sep=r"\s+",
    header=None
)

print("Dataset Shape:", df.shape)



# 2. Create RUL (Remaining Useful Life)
max_cycles = df.groupby(0)[1].max()

df['RUL'] = df.apply(
    lambda row: max_cycles[row[0]] - row[1],
    axis=1
)

print(df[[0, 1, 'RUL']].head())



# 3. Remove Constant Columns

constant_cols = [col for col in df.columns if df[col].nunique() == 1]

print("Constant Columns:", constant_cols)

df_clean = df.drop(columns=constant_cols)

print("Clean Dataset Shape:", df_clean.shape)



# 4. Prepare Features and Target

X = df_clean.drop(columns=['RUL', 0])  # Remove RUL and Engine ID
y = df_clean['RUL']

print("X Shape:", X.shape)
print("y Shape:", y.shape)



# 5. Train-Test Split

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)


# 6. Feature Scaling

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Scaled Training Shape:", X_train_scaled.shape)
print("Scaled Testing Shape :", X_test_scaled.shape)



# 7. Train Linear Regression Model

from sklearn.linear_model import LinearRegression

model = LinearRegression()

model.fit(X_train_scaled, y_train)

print("Model Training Completed")


# 8. Make Predictions
y_pred = model.predict(X_test_scaled)

print("Predicted:", y_pred[:5])
print("Actual   :", y_test[:5].values)



# 9. Evaluate Model

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import numpy as np

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("\n===== Model Evaluation =====")
print("MAE :", mae)
print("RMSE:", rmse)
print("R²  :", r2)


#Random Forest model
from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

rf_model.fit(X_train_scaled, y_train)
print("Random Forest Training Completed....")

rf_pred = rf_model.predict(X_test_scaled)
print("RF Predicted:",rf_pred[:5])
print("Actual  :",y_test[:5].values)

rf_pred = rf_model.predict(X_test_scaled)

rf_mae = mean_absolute_error(y_test,rf_pred)
rf_rmse = np.sqrt(mean_squared_error(y_test,rf_pred))
rf_r2 = r2_score(y_test,rf_pred)

print("RF MAE :",rf_mae)
print("RF RMSE :",rf_rmse)
print("RF R² :",rf_r2)

train_pred = rf_model.predict(X_train_scaled)

train_mae = mean_absolute_error(y_train, train_pred)
train_rmse = np.sqrt(mean_squared_error(y_train, train_pred))
train_r2 = r2_score(y_train, train_pred)


print("Train MAE :",train_mae)
print("Train RMSE :", train_rmse)
print("Train R² :", train_r2)


#load the random forest model 
import joblib
joblib.dump(rf_model,"random_forest_rul.pkl")

#First Hyperparameter tuing 
rf_model2 = RandomForestRegressor(
    n_estimators = 100,
    max_depth = 10,
    random_state = 42
)

rf_model2.fit(X_train_scaled, y_train)

rf2_pred = rf_model2.predict(X_test_scaled)

rf2_mae = mean_absolute_error(y_test, rf2_pred)
rf2_rmse = np.sqrt(mean_squared_error(y_test,rf2_pred))
rf2_r2 = r2_score(y_test, rf2_pred)

print("RF2 MAE :",rf2_mae)
print("RF2 RMSE :",rf2_rmse)
print("RF2 R² :",rf2_r2)

train_pred2 = rf_model2.predict(X_train_scaled)
train_r2_2 = r2_score(y_train, train_pred2)
print("Train R² :",train_r2_2)

🥈 Linear Regression

MAE  = 30.54
RMSE = 39.70
R²   = 0.655

🥇 Random Forest

MAE  = 25.45
RMSE = 35.92
R²   = 0.718